In [0]:
path = "/Volumes/workspace/default/lending_club/Lending_Club_Accepted.csv"
df= spark.read.csv(path, header=True, inferSchema=True)
print("Loaded.")

Loaded.


In [0]:
row_count = df.count()
col_count = len(df.columns)

print(f"Rows: {row_count:,}")
print(f"Columns: {col_count}")

Rows: 2,260,701
Columns: 151


In [0]:
df.groupBy("loan_status").count().orderBy("count", ascending=False).show(truncate=False)

+---------------------------------------------------+-------+
|loan_status                                        |count  |
+---------------------------------------------------+-------+
|Fully Paid                                         |1076751|
|Current                                            |878317 |
|Charged Off                                        |268558 |
|Late (31-120 days)                                 |21467  |
|In Grace Period                                    |8436   |
|Late (16-30 days)                                  |4349   |
|Does not meet the credit policy. Status:Fully Paid |1988   |
|Does not meet the credit policy. Status:Charged Off|761    |
|Default                                            |40     |
|NULL                                               |33     |
|Oct-2015                                           |1      |
+---------------------------------------------------+-------+



In [0]:
# Default Rate based on groups from A to G

from pyspark.sql.functions import col, when, count, avg

(df
 .filter(col("loan_status").isin("Fully Paid", "Charged Off", "Default"))
 .withColumn("is_bad", when(col("loan_status") != "Fully Paid", 1).otherwise(0))
 .groupBy("grade")
 .agg(count("*").alias("loans"), avg("is_bad").alias("default_rate"))
 .orderBy("grade")
 .show()
)

+-----+------+--------------------+
|grade| loans|        default_rate|
+-----+------+--------------------+
|    A|235095|0.060426636040749486|
|    B|392747| 0.13386480355037722|
|    C|381694|  0.2244127494799499|
|    D|200966|  0.3038673208403411|
|    E| 93656|  0.3848231827111984|
|    F| 32059| 0.45204154839514643|
|    G|  9132| 0.49934296977660975|
+-----+------+--------------------+



In [0]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- funded_amnt: double (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: string 

In [0]:
df.select("loan_status", "grade", "issue_d", "loan_amnt", "int_rate", "annual_inc", "dti").show(10, truncate=False)

+-----------+-----+--------+---------+--------+----------+-----+
|loan_status|grade|issue_d |loan_amnt|int_rate|annual_inc|dti  |
+-----------+-----+--------+---------+--------+----------+-----+
|Fully Paid |C    |Dec-2015|3600.0   |13.99   |55000.0   |5.91 |
|Fully Paid |C    |Dec-2015|24700.0  |11.99   |65000.0   |16.06|
|Fully Paid |B    |Dec-2015|20000.0  |10.78   |63000.0   |10.78|
|Current    |C    |Dec-2015|35000.0  |14.85   |110000.0  |17.06|
|Fully Paid |F    |Dec-2015|10400.0  |22.45   |104433.0  |25.37|
|Fully Paid |C    |Dec-2015|11950.0  |13.44   |34000.0   |10.2 |
|Fully Paid |B    |Dec-2015|20000.0  |9.17    |180000.0  |14.67|
|Fully Paid |B    |Dec-2015|20000.0  |8.49    |85000.0   |17.61|
|Fully Paid |A    |Dec-2015|10000.0  |6.49    |85000.0   |13.07|
|Fully Paid |B    |Dec-2015|8000.0   |11.48   |42000.0   |34.8 |
+-----------+-----+--------+---------+--------+----------+-----+
only showing top 10 rows


In [0]:
# Renaming charged off rows as bad and fully paid as 0 while removing the rest
from pyspark.sql.functions import when, col

df = df.withColumn(
    "is_bad",
    when(col("loan_status").isin("Charged Off", "Default"), 1)
    .when(col("loan_status") == "Fully Paid", 0)
    .otherwise(None)
)

In [0]:

df.groupBy("loan_status", "is_bad").count().orderBy("loan_status").show(truncate=False)

+---------------------------------------------------+------+-------+
|loan_status                                        |is_bad|count  |
+---------------------------------------------------+------+-------+
|NULL                                               |NULL  |33     |
|Charged Off                                        |1     |268558 |
|Current                                            |NULL  |878317 |
|Default                                            |1     |40     |
|Does not meet the credit policy. Status:Charged Off|NULL  |761    |
|Does not meet the credit policy. Status:Fully Paid |NULL  |1988   |
|Fully Paid                                         |0     |1076751|
|In Grace Period                                    |NULL  |8436   |
|Late (16-30 days)                                  |NULL  |4349   |
|Late (31-120 days)                                 |NULL  |21467  |
|Oct-2015                                           |NULL  |1      |
+---------------------------------

In [0]:
df_model = df.filter(col("is_bad").isNotNull())
print("Modellable rows:", df_model.count())

Modellable rows: 1345349


In [0]:
# Avg default rate for is_bad
from pyspark.sql.functions import avg
df_model.select(avg("is_bad").alias("overall_default_rate")).show()

+--------------------+
|overall_default_rate|
+--------------------+
| 0.19964931032765476|
+--------------------+



In [0]:
#  Default rate by year
from pyspark.sql.functions import split

df_model = df_model.withColumn("issue_year", split(col("issue_d"), "-").getItem(1).cast("int"))

df_model.groupBy("issue_year").agg(
    avg("is_bad").alias("default_rate"),
    count("*").alias("loans")
).orderBy("issue_year").show()

+----------+-------------------+------+
|issue_year|       default_rate| loans|
+----------+-------------------+------+
|      2007|0.17928286852589642|   251|
|      2008|0.15813060179257363|  1562|
|      2009|0.12595419847328243|  4716|
|      2010| 0.1289008321775312| 11536|
|      2011|0.15178859168546568| 21721|
|      2012| 0.1619727546985965| 53367|
|      2013|0.15595976380522833|134804|
|      2014| 0.1844977431948472|223103|
|      2015|0.20184798093437537|375545|
|      2016|0.23285853192541922|293105|
|      2017|0.23132984095298278|169321|
|      2018|0.15756951596292482| 56318|
+----------+-------------------+------+



In [0]:
from pyspark.sql.functions import col, count, when

missing = df_model.select([
    (count(when(col(c).isNull(), c)) / df_model.count()).alias(c)
    for c in df_model.columns
])
missing_pd = missing.toPandas().T
missing_pd.columns = ["pct_missing"]
print(missing_pd.sort_values("pct_missing", ascending=False).head(40))

                                            pct_missing
member_id                                      1.000000
next_pymnt_d                                   0.999805
orig_projected_additional_accrued_interest     0.997203
hardship_last_payment_amount                   0.995721
hardship_payoff_balance_amount                 0.995721
hardship_dpd                                   0.995718
hardship_loan_status                           0.995716
hardship_start_date                            0.995716
hardship_end_date                              0.995715
hardship_length                                0.995714
payment_plan_start_date                        0.995713
hardship_amount                                0.995710
deferral_term                                  0.995704
hardship_status                                0.995696
hardship_reason                                0.995690
hardship_type                                  0.995684
sec_app_mths_since_last_major_derog            0

In [0]:
print(missing_pd.columns.tolist())
missing_pd.head()

['pct_missing']


,pct_missing
id,0.0
member_id,1.0
loan_amnt,0.0
funded_amnt,0.0
funded_amnt_inv,0.0


In [0]:
# Choosing columns to drop based on index as column names dindt get recognized
high_missing = missing_pd[missing_pd["pct_missing"] > 0.95].index.tolist()
print(len(high_missing), "columns to drop:")
print(high_missing)

38 columns to drop:
['member_id', 'next_pymnt_d', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high', 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog', 'hardship_type', 'hardship_reason', 'hardship_status', 'deferral_term', 'hardship_amount', 'hardship_start_date', 'hardship_end_date', 'payment_plan_start_date', 'hardship_length', 'hardship_dpd', 'hardship_loan_status', 'orig_projected_additional_accrued_interest', 'hardship_payoff_balance_amount', 'hardship_last_payment_amount', 'debt_settlement_flag_date', 'settlement_status', 'settlement_date', 'settlement_amount', 'settlement_percentage', 'settlement_term']


In [0]:
leakage_cols = [
    "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt", "last_credit_pull_d",
    "last_fico_range_high", "last_fico_range_low",
    "out_prncp", "out_prncp_inv",
    "funded_amnt", "funded_amnt_inv",   # post-approval, not application-time
    "issue_d",  # keep issue_year (extracted Day 2), drop the raw timestamp later if you want
]

In [0]:
cols_to_drop = list(set(high_missing + leakage_cols))
df_clean = df_model.drop(*cols_to_drop)
print("Columns before:", len(df_model.columns), "→ after:", len(df_clean.columns))

Columns before: 152 → after: 97


In [0]:
# Changing int_rate to double
from pyspark.sql.functions import col, regexp_replace, trim, when

df_clean = df_clean.withColumn(
    "int_rate",
    regexp_replace(col("int_rate"), "%", "").cast("double")
)

In [0]:
# Changing revol_util to double
df_clean = df_clean.withColumn(
    "revol_util",
    regexp_replace(col("revol_util"), "%", "").cast("double")
)

In [0]:
# Changing term to int
df_clean = df_clean.withColumn(
    "term_months",
    regexp_replace(trim(col("term")), " months", "").cast("int")
).drop("term")


In [0]:
# Changing emp_length to int
df_clean = df_clean.withColumn(
    "emp_length_years",
    when(col("emp_length") == "< 1 year", 0)
    .when(col("emp_length") == "10+ years", 10)
    .otherwise(regexp_replace(col("emp_length"), " years?", "").cast("int"))
).drop("emp_length")

In [0]:
# Checking the columns
df_clean.select("int_rate", "revol_util", "term_months", "emp_length_years").show(10)
df_clean.printSchema()

+--------+----------+-----------+----------------+
|int_rate|revol_util|term_months|emp_length_years|
+--------+----------+-----------+----------------+
|   13.99|      29.7|         36|              10|
|   11.99|      19.2|         36|              10|
|   10.78|      56.2|         60|              10|
|   22.45|      64.5|         60|               3|
|   13.44|      68.4|         36|               4|
|    9.17|      84.5|         36|              10|
|    8.49|       5.7|         36|              10|
|    6.49|      34.5|         36|               6|
|   11.48|      39.1|         36|              10|
|   12.88|      67.2|         36|               3|
+--------+----------+-----------+----------------+
only showing top 10 rows
root
 |-- id: string (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: stri

In [0]:
df_clean = df_clean.withColumn(
    "has_public_record",
    when(col("mths_since_last_record").isNotNull(), 1).otherwise(0)
)

In [0]:
df_clean = df_clean.withColumn(
    "has_prior_delinq",
    when(col("mths_since_last_delinq").isNotNull(), 1).otherwise(0)
)

In [0]:
# Changing issue_d to issue_year

from pyspark.sql.functions import expr

df_clean = df_clean.withColumn(
    "int_rate",
    expr("try_cast(regexp_replace(int_rate, '%', '') AS DOUBLE)")
)

df_clean = df_clean.withColumn(
    "revol_util",
    expr("try_cast(regexp_replace(revol_util, '%', '') AS DOUBLE)")
)

df_clean = df_clean.withColumn(
    "term_months",
    expr("try_cast(regexp_replace(trim(term), ' months', '') AS INT)")
).drop("term")

In [0]:
# Changing emp_length to int
from pyspark.sql.functions import col, when, regexp_replace

df_clean = df_clean.withColumn(
    "emp_length_years",
    when(col("emp_length") == "< 1 year", 0)
    .when(col("emp_length") == "10+ years", 10)
    .otherwise(expr("try_cast(regexp_replace(emp_length, ' years?', '') AS INT)"))
).drop("emp_length")

In [0]:

from pyspark.sql.functions import col, when, regexp_replace, expr

# Start fresh from df_model + drop list
df_clean = df_model.drop(*cols_to_drop)

# Fix all four columns in one go
df_clean = (
    df_clean
    .withColumn("int_rate", expr("try_cast(regexp_replace(int_rate, '%', '') AS DOUBLE)"))
    .withColumn("revol_util", expr("try_cast(regexp_replace(revol_util, '%', '') AS DOUBLE)"))
    .withColumn("term_months", expr("try_cast(regexp_replace(trim(term), ' months', '') AS INT)"))
    .drop("term")
    .withColumn(
        "emp_length_years",
        when(col("emp_length") == "< 1 year", 0)
        .when(col("emp_length") == "10+ years", 10)
        .otherwise(expr("try_cast(regexp_replace(emp_length, ' years?', '') AS INT)"))
    )
    .drop("emp_length")
    .withColumn("has_public_record", when(col("mths_since_last_record").isNotNull(), 1).otherwise(0))
    .withColumn("has_prior_delinq", when(col("mths_since_last_delinq").isNotNull(), 1).otherwise(0))
)

print(df_clean.columns)

['id', 'loan_amnt', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'home_ownership', 'annual_inc', 'verification_status', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m', 'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rc

In [0]:
df_clean.filter(col("int_rate").isNull()).count()

0

In [0]:
# Saving the cleaned data as lc_analytical
df_clean.write.mode("overwrite").saveAsTable("workspace.default.lc_analytical")